# CSE 291 / DSC 291 PA3 — Speculative Decoding

In this notebook you will implement and benchmark a single-sequence (batch=1) speculative decoder.

Recap of the algorithm:

1. A small **draft** model proposes `k` tokens autoregressively starting from the current context.
2. The large **target** model verifies the proposal in **one** forward pass (a single batched pass over the `L + k` length sequence).
3. Tokens are accepted greedily up to the first mismatch with the target's argmax. After the first mismatch, the target's own next token is appended and the loop restarts.

Default model pair (public weights, runs on any GPU with >=4 GB VRAM):

- target: `EleutherAI/pythia-1.4b-deduped`
- draft:  `EleutherAI/pythia-160m-deduped`

If you don't have GPU access, the same code paths run on CPU but you won't see a meaningful speedup.

## Setup

In [1]:
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/ksheersagaragrawal/Desktop/SP26/DSC 291A - MLSystems/cse-dsc291-s26-pa/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Speculative Decoder

In [2]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """Initialize the speculative decoder with a target and a draft model."""
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        assert self.target_tokenizer.get_vocab() == self.draft_tokenizer.get_vocab(), (
            "Target and draft must share a vocabulary"
        )

    def _pick_dtype(self):
        if self.device.startswith("cuda") and torch.cuda.is_available():
            return torch.float16
        return torch.float32

    def initialize_target_model(self, model_name: str):
        """Load the larger target model with caching enabled."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        dtype = self._pick_dtype()
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
        )
        model.to(self.device)
        model.eval()
        model.config.use_cache = True
        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """Load the smaller draft model."""
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        dtype = self._pick_dtype()
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
        )
        model.to(self.device)
        model.eval()
        model.config.use_cache = True
        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                             num_speculative_tokens: int = 10) -> torch.Tensor:
        """
        Generate `num_speculative_tokens` draft tokens with the draft model.

        Args:
            input_ids: Input token IDs (tensor of shape [1, seq_len]).
            attention_mask: Corresponding attention mask.
            num_speculative_tokens: Number of tokens to speculate.

        Returns:
            Tensor of shape [1, num_speculative_tokens] containing the draft tokens.
        """
        generated = []
        with torch.no_grad():
            outputs = self.draft_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=True,
                return_dict=True,
            )
            past_key_values = outputs.past_key_values
            next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
            generated.append(next_token)

            for _ in range(1, num_speculative_tokens):
                outputs = self.draft_model(
                    input_ids=next_token,
                    attention_mask=torch.ones_like(next_token, device=next_token.device),
                    use_cache=True,
                    past_key_values=past_key_values,
                    return_dict=True,
                )
                past_key_values = outputs.past_key_values
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                generated.append(next_token)

        return torch.cat(generated, dim=1)

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                               attention_mask: torch.Tensor) -> Tuple[List[int], int]:
        """
        Vectorized verification: verify all draft tokens in one forward pass using the target model.

        Args:
            input_ids: The current input token IDs (shape [1, L]).
            draft_tokens: Draft tokens from the draft model (shape [1, k]).
            attention_mask: The current attention mask for input_ids.

        Returns:
            accepted_tokens: List of accepted token IDs.
            accepted_position: Index of the first rejected token (if all accepted, equals draft_tokens.shape[1]).
        """
        self._last_verification_target_token = None
        with torch.no_grad():
            combined_ids = torch.cat([input_ids, draft_tokens], dim=1)
            combined_mask = torch.cat([
                attention_mask,
                torch.ones_like(draft_tokens, device=attention_mask.device),
            ], dim=1)
            outputs = self.target_model(
                input_ids=combined_ids,
                attention_mask=combined_mask,
                use_cache=False,
                return_dict=True,
            )
            logits = outputs.logits

        prompt_length = input_ids.shape[1]
        accepted_tokens = []
        accepted_position = 0
        target_next_token = None

        for draft_index in range(draft_tokens.shape[1]):
            target_logits = logits[:, prompt_length + draft_index - 1, :]
            target_token = torch.argmax(target_logits, dim=-1, keepdim=True)
            target_next_token = target_token
            if target_token.item() == draft_tokens[0, draft_index].item():
                accepted_tokens.append(target_token.item())
                accepted_position += 1
            else:
                self._last_verification_target_token = target_token
                break

        if accepted_position == draft_tokens.shape[1]:
            final_logits = logits[:, prompt_length + draft_tokens.shape[1] - 1, :]
            self._last_verification_target_token = torch.argmax(final_logits, dim=-1, keepdim=True)
        elif self._last_verification_target_token is None and target_next_token is not None:
            self._last_verification_target_token = target_next_token

        return accepted_tokens, accepted_position

    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                          num_speculative_tokens: int = 8) -> str:
        """
        Main speculative decoding algorithm with vectorized verification.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate (excluding prompt).
            num_speculative_tokens: Number of tokens to speculate per iteration.

        Returns:
            Generated text.
        """
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        total_tokens_generated = prompt_length
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        start_time = time.time()

        eos_token_id = self.target_tokenizer.eos_token_id

        while total_tokens_generated - prompt_length < max_tokens:
            remaining = max_tokens - (total_tokens_generated - prompt_length)
            k = min(num_speculative_tokens, remaining)
            if k <= 0:
                break

            draft_tokens = self.generate_draft_tokens(input_ids, attention_mask, num_speculative_tokens=k)
            total_draft_tokens_proposed += draft_tokens.shape[1]

            accepted_tokens, accepted_position = self.verify_tokens_vectorized(
                input_ids, draft_tokens, attention_mask
            )
            total_draft_tokens_accepted += accepted_position

            if accepted_position > 0:
                accepted_tensor = torch.tensor(accepted_tokens, device=self.device, dtype=input_ids.dtype).unsqueeze(0)
                input_ids = torch.cat([input_ids, accepted_tensor], dim=1)
                attention_mask = torch.cat([
                    attention_mask,
                    torch.ones_like(accepted_tensor, device=self.device),
                ], dim=1)
                total_tokens_generated += accepted_position

            if total_tokens_generated - prompt_length >= max_tokens:
                break

            target_token = self._last_verification_target_token
            if target_token is None:
                break
            input_ids = torch.cat([input_ids, target_token], dim=1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones_like(target_token, device=self.device),
            ], dim=1)
            total_tokens_generated += 1

            if target_token.item() == eos_token_id:
                break

        elapsed_time = time.time() - start_time
        acceptance_rate = total_draft_tokens_accepted / total_draft_tokens_proposed if total_draft_tokens_proposed > 0 else 0

        print(f"Generated {total_tokens_generated - prompt_length} tokens in {elapsed_time:.2f} seconds")
        print(f"Tokens per second: {(total_tokens_generated - prompt_length) / elapsed_time:.2f}")
        print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        compare_baseline: bool = True,
    ) -> Dict:
        results = {
            "speculative": {"times": [], "tokens_per_second": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None,
        }

        for _ in range(num_runs):
            t0 = time.time()
            output = self.speculative_decode(prompt, max_tokens=max_tokens)
            elapsed = time.time() - t0
            prompt_len = len(self.target_tokenizer(prompt)["input_ids"])
            output_tokens = len(self.target_tokenizer.encode(output)) - prompt_len
            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(output_tokens / elapsed)

        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                t0 = time.time()
                with torch.no_grad():
                    output_ids = self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_length=input_ids.shape[1] + max_tokens,
                        do_sample=False,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                    )
                elapsed = time.time() - t0
                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(output_tokens / elapsed)

        for method in results:
            if results[method] is not None:
                results[method]["avg_time"] = sum(results[method]["times"]) / num_runs
                results[method]["avg_tokens_per_second"] = (
                    sum(results[method]["tokens_per_second"]) / num_runs
                )
        if compare_baseline:
            results["speedup"] = (
                results["baseline"]["avg_time"] / results["speculative"]["avg_time"]
            )
            results["latency_reduction"] = (
                1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]
            ) * 100
        return results

## Test

In [ ]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"
draft_model_name = "EleutherAI/pythia-160m-deduped"

decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

test_prompts = [
    "The future of artificial intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}: {prompt}")
    results = decoder.benchmark(prompt=prompt, max_tokens=100, num_runs=3, compare_baseline=True)
    print(f"  Speculative: {results['speculative']['avg_time']:.2f}s, "
          f"{results['speculative']['avg_tokens_per_second']:.2f} tok/s")
    if results['baseline'] is not None:
        print(f"  Baseline:    {results['baseline']['avg_time']:.2f}s, "
              f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
        print(f"  Speedup: {results['speedup']:.2f}x  |  Latency reduction: {results['latency_reduction']:.2f}%")

## Bonus 3.B — Tree speculation or n-gram lookup decoding (10 pts)

Implement one stronger speculative-decoding variant and benchmark it
against the baseline:

- **Tree / multi-branch speculation** (Medusa / EAGLE-2 style): verify
  several candidate continuations in a single target forward pass.
- **N-gram lookup decoding** (Prompt Lookup Decoding): draft the next
  tokens from an n-gram cache built over the running sequence instead of
  (or in addition to) the draft model.

Re-run the benchmark with your bonus decoder and report the speedup and
acceptance rate in your write-up. See the bonus rubric in `../README.md`.

In [ ]:
# Bonus implementation goes here.
# Re-run the benchmark above with your bonus decoder and copy the numbers into
# your report.